# Inspect a transient on the Rubin Science Platform

Start with a synthetic light curve, then try the same dashboard on a candidate
from your own filtered LSDB catalog.

Open **Notebook Aspect** at [data.lsst.cloud](https://data.lsst.cloud/) and open
this notebook in JupyterLab. See Rubin’s [JupyterLab guide](https://dp2.lsst.io/tutorials/notebook/101/notebook-101-1.html)
and [LSDB tutorial](https://dp2.lsst.io/tutorials/notebook/102/notebook-102-6.html).

Install the package in your notebook environment first. In the JupyterLab
terminal, from the root of your repository checkout:

```sh
python -m pip install .
```

For optional catalog lookups, install with `python -m pip install '.[archives]'`.
If the import fails after installing, check that the terminal and notebook are
using the same Python environment, then restart the kernel.

In [ ]:
import pandas as pd

from rubin_transient_dashboard import inspect_transient

## 1. Try the dashboard with synthetic data

Run the next two cells without changing anything. The synthetic candidate has
one row with its ID and coordinates, and one photometry row per visit and band.
The sky position is illustrative; these cells don't query any archives or need
Rubin data access.

In [ ]:
synthetic_id = 100000000000000001
synthetic_candidates = pd.DataFrame({
    "diaObjectId": [synthetic_id], "ra": [53.0], "dec": [-28.0],
})

visits = [
    (-7, 15), (-3, -30), (-0.21, 780), (-0.11, 890), (-0.02, 950),
    (0.08, 870), (0.22, 760), (2, 375), (5, 105), (9, -25),
]
synthetic_photometry = pd.DataFrame(
    [(synthetic_id, 61000 + day + offset, band, scale * flux, 35)
     for band, scale, offset in [("g", 0.65, 0.04), ("r", 0.85, 0.), ("i", 1., 0.02)]
     for day, flux in visits],
    columns=["diaObjectId", "midpointMjdTai", "band",
             "psfDiffFlux", "psfDiffFluxErr_corrected"],
)

In [ ]:
synthetic = inspect_transient(
    synthetic_id,
    object_table=synthetic_candidates,
    lightcurves=synthetic_photometry,
)

synthetic["light_curve_peak"]

The left panel shows all visits. The right panel selects visits within
half a day of the largest absolute difference-flux measurement, across all
bands, and fits its axes to the selected data. The returned peak metadata
identifies the visit used to center the zoom.

## 2. Inspect your own filtered LSDB catalog

Set `filtered_catalog` to an open LSDB catalog from your workflow. If the
selection lives in another notebook, recreate it in this kernel or save it
as a HATS catalog and set `catalog_path` to that directory. A pandas table of
IDs and coordinates alone is not a photometry catalog.

The catalog needs `diaObjectId`, `ra`, and `dec` (ICRS degrees), plus the
nested column `diaObjectForcedSource` with `midpointMjdTai`, `band`,
`psfDiffFlux`, and `psfDiffFluxErr_corrected`. If you opened the catalog
with a column projection, include those nested fields.

Set `candidate_id` to inspect a particular ID. Leave it as `None` to use the
first available row. Keep IDs as integers or digit strings to avoid rounding.
The code brings only one candidate's top-level coordinates into pandas, then
retrieves its light curve using those coordinates and its exact ID. Finding an
ID can still scan the selected partitions.

This section skips itself until you supply a catalog, so **Run All** is also
safe with just the synthetic example.

In [ ]:
filtered_catalog = None  # e.g. the filtered LSDB catalog created in this kernel
catalog_path = None      # or a path to your saved HATS catalog
candidate_id = None      # or an exact integer ID / digit string

# Start with light curves only. Enable panels after checking data-sharing rules.
panels = ()              # e.g. ("rubin",) or ("legacy", "skybot", "ned")

In [ ]:
if filtered_catalog is None and catalog_path is not None:
    import lsdb
    filtered_catalog = lsdb.open_catalog(catalog_path)

inspection = None
if filtered_catalog is None:
    print("Set filtered_catalog or catalog_path to inspect a Rubin candidate.")
else:
    coords = filtered_catalog.to_dask_dataframe()[["diaObjectId", "ra", "dec"]]
    if candidate_id is not None:
        if not str(candidate_id).isascii() or not str(candidate_id).isdigit():
            raise ValueError("Use an integer ID or a decimal digit string")
        coords = coords[coords["diaObjectId"] == int(candidate_id)]
    candidate = coords.head(1, npartitions=-1)
    if candidate.empty:
        raise LookupError("No matching candidate in your filtered catalog")

    inspection = inspect_transient(
        candidate["diaObjectId"].iloc[0],
        object_table=candidate,
        catalog=filtered_catalog,
        panels=panels,
    )

### Add context when you need it

- `("rubin",)` adds a Rubin coadd cutout if your kernel can access the Butler
  repository. The coadd is static context, not a difference image.
- `"legacy"` shows the Legacy Survey cutout; `"ned"`, `"simbad"`, and `"tns"`
  look for nearby catalog entries (TNS displays a link if credentials are absent).
- `"skybot"` looks for known Solar System objects at the peak visit using
  Rubin's observatory code. A nearby prediction does not establish contamination.

Before enabling external services, check whether you can share the candidate's
coordinates and, for SkyBoT, its observation time. For any query failure,
look in `inspection["errors"]`. An empty successful search is different from
a failed query. See the [README](../README.md) for settings and environment
requirements.